# Hotel Prices in Yerevan
## Arno Barzegarnazari
## DS227B - Business Analytics for Data Science
## Final Project Analysis Notebook


In [ ]:
# -------------
# Importing Libraries and Files
# -------------

import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -------------
# Loading the Dataset
# -------------

file_path = '/content/drive/MyDrive/Business Analytics/Yerevan-Hotels.csv'
df = pd.read_csv(file_path)

print("Initial Shape:", df.shape)
print("Columns:", df.columns.tolist())

Initial Shape: (120, 13)
Columns: ['Hotel Names', 'Star Rating', 'Rating', 'Free Parking', 'Fitness Centre', 'Spa and Wellness Centre', 'Airport Shuttle', 'Staff', 'Facilities', 'Location', 'Comfort', 'Cleanliness', 'Price Per Day($)']


In [ ]:
# -------------
# Dataset Inspection
# -------------

print(df.head())
print(df.info())
print("\nMissing values (counts):\n", df.isnull().sum())

                       Hotel Names  Star Rating  Rating Free Parking  \
0                     Kenut Hostel          NaN     9.7          Yes   
1                    Kantar Hostel          NaN     9.3          Yes   
2               Sweet Sleep hostel          NaN     9.5          Yes   
3  Royal Boutique Hotel on Kievyan          NaN     7.4          Yes   
4                       Areg Hotel          3.0     8.2           No   

  Fitness Centre Spa and Wellness Centre Airport Shuttle  Staff  Facilities  \
0             No                      No              No    9.9         9.8   
1             No                      No             Yes    9.7         9.4   
2             No                      No              No    9.8         9.5   
3             No                      No             Yes    8.1         7.3   
4             No                      No              No    9.1         8.0   

   Location  Comfort  Cleanliness  Price Per Day($)  
0       8.8      9.7          9.8     

In [ ]:
# -------------
# Data Cleaning
# -------------

df = df.drop_duplicates()
df = df.dropna(subset=["Price Per Day($)"])
df = df.fillna(df.mean(numeric_only=True))

df["Free Parking"] = df["Free Parking"].replace({"Yes": 1, "No":0})
df["Fitness Centre"] = df["Fitness Centre"].replace({"Yes": 1, "No":0})
df["Spa and Wellness Centre"] = df["Spa and Wellness Centre"].replace({"Yes": 1, "No":0})
df["Airport Shuttle"] = df["Airport Shuttle"].replace({"Yes": 1, "No":0})

print(df.shape)
df.head()

(120, 13)


/tmp/ipython-input-2133906652.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Free Parking"] = df["Free Parking"].replace({"Yes": 1, "No":0})
/tmp/ipython-input-2133906652.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Fitness Centre"] = df["Fitness Centre"].replace({"Yes": 1, "No":0})
/tmp/ipython-input-2133906652.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in 

,Hotel Names,Star Rating,Rating,Free Parking,Fitness Centre,Spa and Wellness Centre,Airport Shuttle,Staff,Facilities,Location,Comfort,Cleanliness,Price Per Day($)
0,Kenut Hostel,3.772152,9.7,1,0,0,0,9.9,9.8,8.8,9.7,9.8,30.0
1,Kantar Hostel,3.772152,9.3,1,0,0,1,9.7,9.4,9.7,9.2,9.2,15.0
2,Sweet Sleep hostel,3.772152,9.5,1,0,0,0,9.8,9.5,8.9,9.4,9.5,20.0
3,Royal Boutique Hotel on Kievyan,3.772152,7.4,1,0,0,1,8.1,7.3,8.3,7.5,7.8,26.0
4,Areg Hotel,3.000000,8.2,0,0,0,0,9.1,8.0,8.3,8.1,8.2,29.0


In [ ]:
# -------------
# Outlier Detection
# -------------

Q1 = df["Price Per Day($)"].quantile(0.25)
Q3 = df["Price Per Day($)"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

df = df[(df["Price Per Day($)"] >= lower_bound) &
 (df["Price Per Day($)"] <= upper_bound)]

print("Shape after outlier removal:", df.shape)

Shape after outlier removal: (118, 13)


In [ ]:
# -------------
# Multiple Linear Regression
# -------------

features = ["Star Rating", "Rating", "Free Parking", "Fitness Centre",
    "Spa and Wellness Centre", "Airport Shuttle", "Staff",
    "Facilities", "Location", "Comfort", "Cleanliness"]

x = df[features]
y = df["Price Per Day($)"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

coeffs = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_
})

x2 = sm.add_constant(x)
model_sm = sm.OLS(y, x2).fit()


coeffs
print(model_sm.summary())

R2: 0.30678860543227127
MSE: 1134.7939729863976
                            OLS Regression Results                            
Dep. Variable:       Price Per Day($)   R-squared:                       0.481
Model:                            OLS   Adj. R-squared:                  0.427
Method:                 Least Squares   F-statistic:                     8.934
Date:                Tue, 02 Dec 2025   Prob (F-statistic):           4.45e-11
Time:                        12:09:11   Log-Likelihood:                -561.17
No. Observations:                 118   AIC:                             1146.
Df Residuals:                     106   BIC:                             1180.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------

In [ ]:
# -------------
# Lasso and Ridge Regression
# -------------

lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(alpha=0.1))
])

ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=0.1))
])

lasso_pipeline.fit(x_train, y_train)
ridge_pipeline.fit(x_train, y_train)

lasso_pred = lasso_pipeline.predict(x_test)
ridge_pred = ridge_pipeline.predict(x_test)

print("Lasso R2:", r2_score(y_test, lasso_pred))
print("Ridge R2:", r2_score(y_test, ridge_pred))

Lasso R2: 0.3623883724632536
Ridge R2: 0.3431721156159626


In [ ]:
# -------------
# K-Means Clustering
# -------------

cluster_features = [
    "Star Rating", "Rating", "Free Parking", "Fitness Centre",
    "Spa and Wellness Centre", "Airport Shuttle", "Staff",
    "Facilities", "Location", "Comfort", "Cleanliness"
]

x_clust = df[cluster_features]

kmeans = KMeans(n_clusters=3, random_state=42)
df["Cluster"] = kmeans.fit_predict(x_clust)

df[["Hotel Names", "Cluster"]].head()
df.groupby("Cluster")["Price Per Day($)"].mean()

,Price Per Day($)
Cluster,
0,67.032258
1,72.337209
2,110.000000
